In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="multilabel_sigmoid_zero_shot_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "typeform/distilbert-base-uncased-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

classifier = pipeline(
    "zero-shot-classification",
    model=model,
    tokenizer=tokenizer,
    device=device,
    framework="pt",
)

print(model_name)
print(model.config.id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

typeform/distilbert-base-uncased-mnli
{0: 'ENTAILMENT', 1: 'NEUTRAL', 2: 'CONTRADICTION'}


In [3]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

texts = [f"Sentence 1: {s1}\nSentence 2: {s2}" for s1, s2 in zip(sent1, sent2)]
candidate_labels = ["paraphrase", "not paraphrase"]
hypothesis_template = "These two sentences are {}."

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())
print(texts[0])

'The read operation timed out' thrown while requesting HEAD https://huggingface.co/datasets/nyu-mll/glue/resolve/bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/dataset_infos.json
Retrying in 1s [Retry 1/5].


Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
Sentence 1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
Sentence 2: " The foodservice pie business does not fit our long-term growth strategy .


In [4]:
batch_size = 16
outputs = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch_texts = texts[i:i + batch_size]
    batch_outputs = classifier(
        batch_texts,
        candidate_labels=candidate_labels,
        hypothesis_template=hypothesis_template,
        multi_label=True,
        batch_size=batch_size,
    )
    if isinstance(batch_outputs, dict):
        batch_outputs = [batch_outputs]
    outputs.extend(batch_outputs)

paraphrase_scores = []
not_paraphrase_scores = []
preds = []

for out in outputs:
    score_map = {label: float(score) for label, score in zip(out["labels"], out["scores"])}
    para_score = score_map["paraphrase"]
    not_para_score = score_map["not paraphrase"]
    paraphrase_scores.append(para_score)
    not_paraphrase_scores.append(not_para_score)
    preds.append(1 if para_score >= not_para_score else 0)

paraphrase_scores = np.array(paraphrase_scores)
not_paraphrase_scores = np.array(not_paraphrase_scores)
y_pred = np.array(preds)

print("done")

  0%|          | 0/26 [00:00<?, ?it/s]

done


In [ ]:

vault.create_record_list("distilbert-paraphrase_scores-and-prediction", column_names=["prediction", "paraphrase_scores" , "not_paraphrase_scores"])

for i in range(len(y_pred)):
    vault.append_record("distilbert-paraphrase_scores-and-prediction", 
                        {
                            "prediction": y_pred[i],
                            "paraphrase_scores": float(paraphrase_scores[i]),
                            "not_paraphrase_scores": float(not_paraphrase_scores[i]),
                        },
                       input_items = {
                           "glue_mrpc_validation": [i, i + 1],
                       }
                       )

description = "INSERT TEXT HERE ABOUT distilbert-paraphrase_scores-and-predictionn"
embedding = get_embeddings(description)
vault.create_description("distilbert-paraphrase_scores-and-prediction", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("distilbert-paraphrase_scores-and-prediction", cat, embedding, prop)

In [5]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=['not_paraphrase', 'paraphrase'])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.6764705882352942, 'f1': 0.8018018018018018}
                precision    recall  f1-score   support

not_paraphrase       0.43      0.07      0.12       129
    paraphrase       0.69      0.96      0.80       279

      accuracy                           0.68       408
     macro avg       0.56      0.51      0.46       408
  weighted avg       0.61      0.68      0.59       408



In [6]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("score_paraphrase:", float(paraphrase_scores[i]))
    print("score_not_paraphrase:", float(not_paraphrase_scores[i]))

mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))
    print("score_paraphrase:", float(paraphrase_scores[i]))
    print("score_not_paraphrase:", float(not_paraphrase_scores[i]))

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1
score_paraphrase: 0.5612597465515137
score_not_paraphrase: 0.11629904806613922
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 1
score_paraphrase: 0.39619821310043335
score_not_paraphrase: 0.08260387182235718
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 1
score_paraphrase: 0.410314679145813
score_not_paraphrase: 0.129601955

In [7]:
vault.create_record_list("multilabel_sigmoid_zero_shot_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("multilabel_sigmoid_zero_shot_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "distilbert-paraphrase_scores-and-prediction": [0, len(ds)]
                    })

summary

description = "INSERT TEXT HERE ABOUT multilabel_sigmoid_zero_shot_mrpc_summary"
embedding = get_embeddings(description)
vault.create_description("multilabel_sigmoid_zero_shot_mrpc_summary", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("multilabel_sigmoid_zero_shot_mrpc_summary", cat, embedding, prop)


{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'typeform/distilbert-base-uncased-mnli',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.6764705882352942,
 'f1': 0.8018018018018018}

In [ ]:
description = "INSERT TEXT HERE ABOUT multilabel_sigmoid_zero_shot_mrpc process" # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("multilabel_sigmoid_zero_shot_mrpc", description, embedding)

properties = {"INSERT_PROPERTY": "INSERT_CATEGORIES"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("multilabel_sigmoid_zero_shot_mrpc", cat, embedding, prop)